# CoreField battery track — Part 6: inverse PINN on real Li-ion data

**The question.** Parts 4 and 5 established the machinery on synthetic data, where the finite-difference truth solves exactly the PDE the network assumes. Part 3 established a classical inverse that works on *real* NASA cells at 1.44 % relative accuracy, corroborated to 1.14× by an independent EIS measurement. This notebook closes the gap: **does the PINN transfer to real data, and does it add anything the classical fitter cannot give?**

**What is being solved.** The same three unknowns Part 3 recovers — R₀ and the two coefficients of the non-monotonic source shape R(x) = R₀(1 + c₁x + c₂x²), with x the coulomb-counted depth of discharge — from **one** surface temperature channel plus the measured current. h is held at the Part-3 pooled profile-likelihood value of 24.81 W m⁻²K⁻¹, because Part-3 Cell 7h showed {R₀, c₁, c₂, h} is not identifiable from a single channel: with h free the worst relative CRLB standard deviation exceeds 100 %, so a four-parameter fit would produce numbers that look reasonable and mean nothing.

**Three things this notebook cannot do, stated up front.**

1. **The field cannot be validated.** There is no internal thermocouple in a sealed 18650, so the reconstructed interior is unverifiable on this data. It can be checked for physical plausibility and self-consistency, nothing more. Validating it needs an instrumented teardown cell — see §4.
2. **Absolute R₀ inherits an assumed ρc_p.** Part 3 flagged this: ρc_p = 2.75×10⁶ J m⁻³K⁻¹ is **(b)**, taken from Part 2, not measured for these cells. Since q‴ ∝ R₀/ρc_p, absolute values scale with it. The *comparison* against the classical fitter is clean because both use the same value.
3. **This is not a commercial demonstration.** It is a transfer test. The runtime is minutes per cycle of offline optimisation, not a BMS component.

## 1. Pre-registered predictions (G-series)

Registered before the first execution, and deliberately including the possibility that the PINN adds nothing measurable here.

| # | Prediction | Basis |
|---|---|---|
| G1 | The PINN converges on the stratum's first cycle with surface RMSE **≤ 0.30 K**, matching the Part-3 gate | same source model, more flexible solver |
| **G2** | **R₀ lands within ±3 % of the classical 152.99 mΩ** | with Bi ≈ 0.07 the lumped model is nearly exact for an 18650, so the eigenmode correction that mattered in Part 5 should be small here |
| G3 | c₁ and c₂ reproduce the U-shape: **c₁ < 0, c₂ > 0, minimum in x ∈ [0.2, 0.4]** | Part 3 recovered −1.352 / +2.192, minimum at 0.308 |
| G4 | The reconstructed bow at end of discharge is **< 0.5 K** and positive | Bi ≈ 0.07 ⇒ interior only slightly hotter than the surface |
| G5 | R₀ comes out **higher** than the classical value by 0.5–2 % | the lumped form under-estimates by the eigenmode argument of Part 2's B3 check; Part 5 measured that correction at ≈ 1.5 % |

**G5 is the load-bearing one, and it is the only claim here that the PINN is *better* rather than merely equal.** If R₀ moves up by roughly the predicted amount, the eigenmode correction demonstrated on synthetic data in Part 5 has transferred to real cells. If it does not move, the honest conclusion is that at Bi ≈ 0.07 the classical fitter is already sufficient for this geometry, and the PINN's justification on real data rests entirely on the field — which cannot be validated without new hardware.

Either outcome is a result. The second is less flattering and more useful, because it would tell you exactly where the method's value does and does not lie.

In [1]:
# ===== CELL 1 : SETUP — real-cell constants and the Part-3 reference =======
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import time, glob, copy, numpy as np, torch, torch.nn as nn
import torch._utils  # noqa: F401   (Kaggle GPU-image workaround, see Part 5b)
from scipy.optimize import curve_fit
torch.set_num_threads(2)
DEVICE = "cpu"                       # keep CPU: the CPU baseline is the published one
DEV = torch.device(DEVICE if (DEVICE == "cpu" or torch.cuda.is_available()) else "cpu")
print(f"torch {torch.__version__} | device {DEV}")

# 18650 geometry, identical to Parts 2-5 so the comparison is like-for-like
L, RHO_CP, K = 0.018, 2500.0*1100.0, 2.5          # rho*cp is (b), NOT measured
V_CELL = np.pi*0.009**2*0.065
H_FIX  = 24.81                                    # Part-3 pooled profile likelihood
T_ON_NOM = 3000.0

# ---- Part-3 classical reference on B0005 @ 24 C, 168 cycles ---------------
REF = dict(R0_mOhm=152.99, c1=-1.352, c2=2.192, R_end_mOhm=280.39,
           rmse=0.228, rise=15.87, rel_pct=1.44,
           corr_cap_R0=-0.908, EIS_Re_Rct_mOhm=134.32, ratio_thermal_EIS=1.14)
for k_, v in REF.items(): print(f"  ref {k_:18s} {v}")

torch 2.13.0+cpu | device cpu
  ref R0_mOhm            152.99
  ref c1                 -1.352
  ref c2                 2.192
  ref R_end_mOhm         280.39
  ref rmse               0.228
  ref rise               15.87
  ref rel_pct            1.44
  ref corr_cap_R0        -0.908
  ref EIS_Re_Rct_mOhm    134.32
  ref ratio_thermal_EIS  1.14


In [2]:
# ===== CELL 2 : LOAD ONE STRATUM (reuses the Part-3 loader logic) ==========
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "."]
def discover():
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root): continue
        mats = sorted(glob.glob(os.path.join(root, "**", "B0*.mat"), recursive=True))
        if mats: return "mat", os.path.dirname(mats[0]), None
        for m in glob.glob(os.path.join(root, "**", "metadata.csv"), recursive=True):
            d = os.path.dirname(m)
            cd = {}
            for p in glob.glob(os.path.join(d, "**", "*.csv"), recursive=True):
                cd[os.path.dirname(p)] = cd.get(os.path.dirname(p), 0) + 1
            if cd:
                return "csv", max(cd, key=cd.get), m
    return None, None, None

LAYOUT, CYCLE_DIR, META = discover()
print("layout:", LAYOUT, "|", CYCLE_DIR)
BATTERY, AMBIENT = "B0005", 24.0

def load_discharge():
    out = []
    if LAYOUT == "mat":
        from scipy.io import loadmat
        m = loadmat(os.path.join(CYCLE_DIR, BATTERY+".mat"), simplify_cells=True)
        key = [k for k in m if not k.startswith("__")][0]
        for i, c in enumerate(m[key]["cycle"]):
            if c["type"] != "discharge": continue
            d = c["data"]; cap = d.get("Capacity", np.nan)
            out.append(dict(idx=i, t=np.asarray(d["Time"], float),
                            I=np.abs(np.asarray(d["Current_measured"], float)),
                            T=np.asarray(d["Temperature_measured"], float)+273.15,
                            T_amb=float(c.get("ambient_temperature", 24.0))+273.15,
                            cap=float(np.atleast_1d(cap)[0]) if np.size(cap) else np.nan))
    elif LAYOUT == "csv":
        import pandas as pd
        md_ = pd.read_csv(META)
        sel = md_[md_["type"].astype(str).str.lower() == "discharge"]
        if "battery_id" in md_.columns:
            sel = sel[sel["battery_id"].astype(str) == BATTERY]
        if "ambient_temperature" in md_.columns:
            sel = sel[np.isclose(pd.to_numeric(sel["ambient_temperature"],
                                               errors="coerce"), AMBIENT)]
        for i, row in sel.iterrows():
            p = os.path.join(CYCLE_DIR, str(row["filename"]))
            if not os.path.exists(p): continue
            d = pd.read_csv(p)
            try: cap = float(row["Capacity"])
            except Exception: cap = np.nan
            out.append(dict(idx=int(i), t=d["Time"].to_numpy(float),
                            I=np.abs(d["Current_measured"].to_numpy(float)),
                            T=d["Temperature_measured"].to_numpy(float)+273.15,
                            T_amb=float(row.get("ambient_temperature", 24.0))+273.15,
                            cap=cap))
    else:
        raise RuntimeError("attach the NASA dataset first")
    return out

cyc = load_discharge()
print(f"{len(cyc)} discharge cycles for {BATTERY} @ {AMBIENT} C")
c0 = cyc[0]
print(f"cycle 0: {c0['t'][-1]-c0['t'][0]:.0f} s | rise "
      f"{c0['T'].max()-c0['T'][0]:.2f} K | cap {c0['cap']:.3f} Ah")

layout: mat | .\Data_Sets\Li-ion Battery Dataset from NASA PCoE\Battery_DataSet\Battery_DataSet


168 discharge cycles for B0005 @ 24.0 C
cycle 0: 3690 s | rise 14.65 K | cap 1.856 Ah


In [3]:
# ===== CELL 3 : SCALING FOR A REAL CYCLE ===================================
# The unknowns are the SAME three the Part-3 classical fitter recovers:
#   R0  source amplitude at full charge
#   c1, c2  the non-monotonic shape vs depth of discharge, R(x)=R0(1+c1 x+c2 x^2)
# h is FIXED at the Part-3 pooled profile-likelihood value: Cell 7h of Part 3
# showed {R0, c1, c2, h} is not identifiable from one channel (worst CRLB sd
# > 100 % with h free), so freeing it here would produce numbers that look fine
# and mean nothing.
def prep(c, t_ref=None, R_ref=0.150):
    t = c['t'] - c['t'][0]; T = c['T']; I = c['I']
    tref = t[-1] if t_ref is None else t_ref
    cum = np.concatenate([[0.0], np.cumsum(I[:-1]*np.diff(t))])
    x = cum/cum[-1] if cum[-1] > 0 else np.zeros_like(t)
    q3_ref = I.mean()**2*R_ref/V_CELL
    dT_ref = q3_ref*L/(2*H_FIX)
    return dict(t=t, T=T, I=I, x=x, tref=tref, R_ref=R_ref, dT_ref=dT_ref,
                Fo=(K/RHO_CP)*tref/L**2,
                S_ref=q3_ref*tref/(RHO_CP*dT_ref),
                Bi=H_FIX*L/K, T_amb=c['T_amb'], T0=T[0], cap=c['cap'], idx=c['idx'])

P = prep(cyc[0])
print(f"Fo = {P['Fo']:.3f}   S_ref = {P['S_ref']:.3f}   Bi = {P['Bi']:.4f}   "
      f"dT_ref = {P['dT_ref']:.3f} K")
print(f"Bi = {P['Bi']:.4f} -> internal gradient is ~{100*P['Bi']/4:.1f} % of the surface rise")
print("With Bi this small the lumped model is nearly exact for this cell, so the")
print("PINN's parameter advantage should be SMALL here - see the predictions below.")

Fo = 10.354   S_ref = 3.699   Bi = 0.1786   dT_ref = 10.881 K
Bi = 0.1786 -> internal gradient is ~4.5 % of the surface rise
With Bi this small the lumped model is nearly exact for this cell, so the
PINN's parameter advantage should be SMALL here - see the predictions below.


In [4]:
# ===== CELL 4 : NETWORK WITH THREE UNKNOWN SOURCE PARAMETERS ===============
class RealInvNet(nn.Module):
    # theta0 is the KNOWN initial offset (T[0]-T_amb)/dT_ref. Without it the hard
    # IC forces theta(x,0) = 0, i.e. T(x,0) = T_amb exactly -- but a NASA cell
    # begins discharge after a charge step and starts WARM. The network was then
    # structurally forbidden from matching its own first observation, and no
    # amount of data weight can repair that.
    def __init__(s, w=48, d=5, R0_mult=1.0, c1_0=-1.0, c2_0=2.0, theta0=0.0):
        super().__init__()
        s.theta0 = float(theta0)
        layers = [nn.Linear(2, w), nn.Tanh()]
        for _ in range(d-1): layers += [nn.Linear(w, w), nn.Tanh()]
        layers += [nn.Linear(w, 1)]
        s.f = nn.Sequential(*layers)
        s.pR = nn.Parameter(torch.tensor(float(np.log(R0_mult))))
        s.c1 = nn.Parameter(torch.tensor(float(c1_0)))
        s.c2 = nn.Parameter(torch.tensor(float(c2_0)))
    def forward(s, x, t): return s.theta0 + t*s.f(torch.cat([x, t], 1))
    def rR(s): return torch.exp(s.pR)

def derivs(net, x, t):
    x = x.clone().requires_grad_(True); t = t.clone().requires_grad_(True)
    th = net(x, t)
    gx = torch.autograd.grad(th, x, torch.ones_like(th), create_graph=True)[0]
    gxx = torch.autograd.grad(gx, x, torch.ones_like(gx), create_graph=True)[0]
    gt = torch.autograd.grad(th, t, torch.ones_like(th), create_graph=True)[0]
    return th, gx, gxx, gt

def train_real(P, seed=0, adam_ep=3000, lb_it=300, n_pass=3,
               w_bc=100.0, wd1=200.0, wd2=20.0, N=2000, NB=250, log_every=750):
    torch.manual_seed(seed); g = torch.Generator().manual_seed(seed+11)
    th0 = (P['T0'] - P['T_amb'])/P['dT_ref']
    print(f"  IC offset: T[0]-T_amb = {P['T0']-P['T_amb']:.3f} K  "
          f"-> theta0 = {th0:.4f} scaled")
    net = RealInvNet(theta0=th0).to(DEV)
    # observation channel, scaled
    td = torch.tensor(P['t']/P['tref'], dtype=torch.float32).reshape(-1,1).to(DEV)
    ob = torch.tensor((P['T']-P['T_amb'])/P['dT_ref'], dtype=torch.float32).reshape(-1,1).to(DEV)
    z0 = torch.zeros_like(td)
    xt = torch.tensor(P['x'], dtype=torch.float32)
    def x_of_t(tt):   # depth of discharge at scaled time tt
        return torch.from_numpy(np.interp(tt.detach().cpu().numpy().ravel(),
                                          P['t']/P['tref'], P['x'])
                                ).float().reshape(-1,1).to(DEV)
    def terms(xi=None, ti=None, tb=None):
        if xi is None:
            xi = torch.rand(N,1,generator=g).to(DEV); ti = torch.rand(N,1,generator=g).to(DEV)
            tb = torch.rand(NB,1,generator=g).to(DEV)
        _, _, xx, tt = derivs(net, xi, ti)
        xd = x_of_t(ti)
        mult = 1.0 + net.c1*xd + net.c2*xd**2
        Lr = ((tt - P['Fo']*xx - P['S_ref']*net.rR()*mult)**2).mean()
        th0, g0, _, _ = derivs(net, torch.zeros_like(tb), tb)
        thL, gL, _, _ = derivs(net, torch.ones_like(tb), tb)
        Lb = ((g0 - P['Bi']*th0)**2).mean() + ((gL + P['Bi']*thL)**2).mean()
        Ld = ((net(z0, td) - ob)**2).mean()          # ONE channel only
        return Lr, Lb, Ld
    terms.fresh = lambda: (torch.rand(N,1,generator=g).to(DEV),
                           torch.rand(N,1,generator=g).to(DEV),
                           torch.rand(NB,1,generator=g).to(DEV))
    phys = [net.pR, net.c1, net.c2]; pid = {id(p) for p in phys}
    body = [p for p in net.parameters() if id(p) not in pid]
    opt = torch.optim.Adam([{"params": body, "lr": 2e-3}, {"params": phys, "lr": 2e-3}])
    n1 = int(0.4*adam_ep)
    for ep in range(n1):
        Lr, Lb, Ld = terms(); (Lr+w_bc*Lb+wd1*Ld).backward(); opt.step(); opt.zero_grad()
        if (ep+1) % log_every == 0:
            print(f"  P1 {ep+1:5d} Lr {float(Lr.detach()):.3e} Ld {float(Ld.detach()):.3e} "
                  f"R0 {P['R_ref']*float(net.rR().detach())*1e3:.1f} mOhm")
    for gp, lr in zip(opt.param_groups, (1e-3, 5e-5)): gp["lr"] = lr
    for ep in range(n1, adam_ep):
        Lr, Lb, Ld = terms(); (Lr+w_bc*Lb+wd2*Ld).backward(); opt.step(); opt.zero_grad()
        if (ep+1) % log_every == 0:
            print(f"  P2 {ep+1:5d} Lr {float(Lr.detach()):.3e} Ld {float(Ld.detach()):.3e} "
                  f"R0 {P['R_ref']*float(net.rR().detach())*1e3:.1f} mOhm")
    best, bstate = None, None
    for k in range(n_pass):
        xf, tf, tbf = terms.fresh()
        lb = torch.optim.LBFGS(net.parameters(), max_iter=lb_it, tolerance_grad=1e-16,
                               tolerance_change=1e-18, history_size=80,
                               line_search_fn="strong_wolfe")
        def cl():
            lb.zero_grad(); a,b_,c_ = terms(xf,tf,tbf); l = a+w_bc*b_+wd2*c_
            l.backward(); return l
        lb.step(cl)
        a, b_, c_ = terms(); lrf = float(a.detach())
        if best is None or lrf < best:            # truth-free criterion
            best, bstate = lrf, copy.deepcopy(net.state_dict())
        print(f"  pass {k+1}: Lr {lrf:.3e}")
    net.load_state_dict(bstate)
    return net, terms, best

In [5]:
# ===== CELL 5 : RUN ON ONE CYCLE AND COMPARE WITH THE CLASSICAL FITTER =====
net, terms, Lr_best = train_real(P)
R0 = P['R_ref']*float(net.rR().detach())
c1 = float(net.c1.detach()); c2 = float(net.c2.detach())
# reconstructed surface trace -> RMSE against the measurement
with torch.no_grad():
    td = torch.tensor(P['t']/P['tref'], dtype=torch.float32).reshape(-1,1).to(DEV)
    Ts = P['T_amb'] + P['dT_ref']*net(torch.zeros_like(td), td).cpu().numpy().ravel()
rmse = float(np.sqrt(np.mean((Ts-P['T'])**2)))
rise = float(P['T'].max()-P['T'][0])
# internal field: the output the classical fitter cannot produce
with torch.no_grad():
    xs = torch.linspace(0, 1, 61).reshape(-1,1).to(DEV)
    tt = torch.full_like(xs, 1.0)
    prof = P['T_amb'] + P['dT_ref']*net(xs, tt).cpu().numpy().ravel()
bow = float(prof[30]-prof[0])

print(f"\n{'quantity':22s}{'PINN':>12s}{'classical (Part 3)':>22s}")
print(f"{'R0 (mOhm)':22s}{R0*1e3:12.2f}{REF['R0_mOhm']:22.2f}")
print(f"{'c1':22s}{c1:12.3f}{REF['c1']:22.3f}")
print(f"{'c2':22s}{c2:12.3f}{REF['c2']:22.3f}")
print(f"{'R_end (mOhm)':22s}{R0*(1+c1+c2)*1e3:12.2f}{REF['R_end_mOhm']:22.2f}")
print(f"{'surface RMSE (K)':22s}{rmse:12.4f}{REF['rmse']:22.4f}")
print(f"{'relative (%)':22s}{100*rmse/rise:12.2f}{REF['rel_pct']:22.2f}")
print(f"\nR0 vs classical: {100*(R0*1e3/REF['R0_mOhm']-1):+.2f} %")
print(f"core-surface bow at end of discharge: {bow:.4f} K   "
      f"(classical fitter: not available at any accuracy)")
print(f"thermal/EIS ratio: {R0*1e3/REF['EIS_Re_Rct_mOhm']:.2f}x   "
      f"(classical gave {REF['ratio_thermal_EIS']:.2f}x)")

  IC offset: T[0]-T_amb = 0.330 K  -> theta0 = 0.0303 scaled


  P1   750 Lr 5.176e-02 Ld 3.481e-03 R0 142.0 mOhm


  P2  1500 Lr 1.567e-02 Ld 4.368e-03 R0 146.6 mOhm


  P2  2250 Lr 1.270e-02 Ld 4.287e-03 R0 148.0 mOhm


  P2  3000 Lr 8.736e-03 Ld 4.441e-03 R0 149.6 mOhm


  pass 1: Lr 3.134e-03


  pass 2: Lr 2.836e-03


  pass 3: Lr 2.707e-03

quantity                      PINN    classical (Part 3)
R0 (mOhm)                   144.09                152.99
c1                           0.010                -1.352
c2                           0.442                 2.192
R_end (mOhm)                209.34                280.39
surface RMSE (K)            0.6161                0.2280
relative (%)                  4.20                  1.44

R0 vs classical: -5.82 %
core-surface bow at end of discharge: 0.7245 K   (classical fitter: not available at any accuracy)
thermal/EIS ratio: 1.07x   (classical gave 1.14x)


In [6]:
# ===== CELL 5b : LIKE-FOR-LIKE CLASSICAL COMPARATOR ===================
# The Part-3 reference (c1 = -1.352, c2 = +2.192) is a MEDIAN over 168 cycles,
# while the PINN above fitted cycle 0 alone. Comparing the two is only valid if
# cycle-to-cycle shape variation is small -- which Part 3 never quantified.
# This cell runs the Part-3 classical fitter on the SAME cycles from the SAME
# loader, so the comparison becomes like-for-like, and reports the spread.
# Cost: milliseconds per cycle. Run this BEFORE the w_data sweep in Cell 6.

def theta_exp(t, q, h, theta0, L_, rho_cp):
    tau = rho_cp*L_/(2.0*h)
    th = np.empty_like(t); th[0] = theta0
    dt = np.diff(t); E = np.exp(-dt/tau)
    for i in range(dt.size):
        th[i+1] = th[i]*E[i] + (q[i]/rho_cp)*tau*(1.0 - E[i])
    return th

def model_poly(t, R0, coefs, h, I, x, T_amb, T0, L_, V_, rho_cp):
    mult = np.ones_like(x)
    for k, ck in enumerate(coefs, start=1):
        mult = mult + ck*x**k
    return T_amb + theta_exp(t, I**2*R0*mult/V_, h, T0-T_amb, L_, rho_cp)

def fit_poly_cycle(c, order=2, h=H_FIX, maxpts=600):
    t = c['t']-c['t'][0]; y = c['T']; I = c['I']
    if t.size > maxpts:
        k = np.linspace(0, t.size-1, maxpts).astype(int); t, y, I = t[k], y[k], I[k]
    cum = np.concatenate([[0.0], np.cumsum(I[:-1]*np.diff(t))])
    x = cum/cum[-1] if cum[-1] > 0 else np.zeros_like(t)
    lo = [0.020]+[-20.0]*order; hi = [0.300]+[20.0]*order; p0 = [0.09]+[0.5]*order
    def m(tt, *pp):
        return model_poly(tt, pp[0], list(pp[1:1+order]), h, I, x,
                          c['T_amb'], y[0], L, V_CELL, RHO_CP)
    popt, _ = curve_fit(m, t, y, p0=p0, bounds=(lo, hi), method='trf',
                        loss='soft_l1', f_scale=0.5, max_nfev=6000)
    yf = m(t, *popt)
    return dict(R0_mOhm=popt[0]*1e3, c1=popt[1], c2=popt[2],
                R_end=popt[0]*(1+popt[1]+popt[2])*1e3,
                rmse=float(np.sqrt(np.mean((yf-y)**2))),
                rise=float(y.max()-y[0]))

# --- (a) the SAME cycle the PINN used ------------------------------------
c0f = fit_poly_cycle(cyc[0])
print("CYCLE 0 — the correct like-for-like comparator")
print(f"{'quantity':14s}{'PINN':>10s}{'classical(c0)':>15s}{'classical(median168)':>22s}")
for k_, pv, cv, mv in [("R0 mOhm", R0*1e3, c0f['R0_mOhm'], REF['R0_mOhm']),
                       ("c1", c1, c0f['c1'], REF['c1']),
                       ("c2", c2, c0f['c2'], REF['c2']),
                       ("R_end mOhm", R0*(1+c1+c2)*1e3, c0f['R_end'], REF['R_end_mOhm']),
                       ("RMSE K", rmse, c0f['rmse'], REF['rmse'])]:
    print(f"{k_:14s}{pv:10.3f}{cv:15.3f}{mv:22.3f}")

# --- (b) shape variation across the whole stratum -------------------------
allf = []
for c in cyc:
    try: allf.append(fit_poly_cycle(c))
    except Exception: pass
import statistics as st
def q(v, p): 
    s = sorted(v); i = (len(s)-1)*p; lo_ = int(i); hi_ = min(lo_+1, len(s)-1)
    return s[lo_] + (s[hi_]-s[lo_])*(i-lo_)
print(f"\nSHAPE VARIATION across {len(allf)} classical fits")
print(f"{'param':10s}{'median':>10s}{'IQR':>18s}{'p5–p95':>20s}{'cycle 0':>10s}")
for nm in ("c1", "c2", "R0_mOhm", "R_end"):
    v = [f[nm] for f in allf]
    print(f"{nm:10s}{st.median(v):10.3f}"
          f"{'['+format(q(v,.25),'.3f')+', '+format(q(v,.75),'.3f')+']':>18s}"
          f"{'['+format(q(v,.05),'.3f')+', '+format(q(v,.95),'.3f')+']':>20s}"
          f"{c0f[nm]:10.3f}")

# --- (c) verdict -----------------------------------------------------------
c1v = [f["c1"] for f in allf]; c2v = [f["c2"] for f in allf]
in5_95 = (q(c1v,.05) <= c1 <= q(c1v,.95)) and (q(c2v,.05) <= c2 <= q(c2v,.95))
near_c0 = abs(c1-c0f['c1']) < 0.3 and abs(c2-c0f['c2']) < 0.5
print()
if near_c0:
    print("VERDICT: the PINN shape matches CYCLE 0's OWN classical fit. The apparent")
    print("25 % discrepancy was an artifact of comparing against a 168-cycle median.")
    print("No w_data problem is demonstrated -> Cell 6 is NOT needed for this reason.")
elif in5_95:
    print("VERDICT: the PINN shape sits inside the cycle-to-cycle p5–p95 band but not")
    print("at cycle 0's own value. Partly artifact, partly real -> run Cell 6, and")
    print("judge it against cycle 0's classical value, not the median.")
else:
    print("VERDICT: the PINN shape lies OUTSIDE the cycle-to-cycle spread. The")
    print("discrepancy is REAL and not a comparator artifact -> Cell 6 is the right")
    print("next step, with the registered prediction as written.")
print()
iqr_c1 = q(c1v,.75)-q(c1v,.25); iqr_c2 = q(c2v,.75)-q(c2v,.25)
print(f"Note for Part 3: shape IQR is c1 {iqr_c1:.3f}, c2 {iqr_c2:.3f}. If these are")
print("wide relative to the medians, Part 3's single median shape needs the spread")
print("reported beside it -- a caveat that applies to the thesis text, not just here.")

CYCLE 0 — the correct like-for-like comparator
quantity            PINN  classical(c0)  classical(median168)
R0 mOhm          144.090        144.352               152.990
c1                 0.010         -1.305                -1.352
c2                 0.442          1.934                 2.192
R_end mOhm       209.338        235.148               280.390
RMSE K             0.616          0.130                 0.228



SHAPE VARIATION across 168 classical fits
param         median               IQR              p5–p95   cycle 0
c1            -1.352  [-1.444, -1.277]    [-1.492, -1.207]    -1.305
c2             2.192    [2.107, 2.252]      [1.920, 2.330]     1.934
R0_mOhm      152.987[139.331, 162.042]  [131.247, 166.762]   144.352
R_end        280.388[236.420, 306.465]  [228.434, 313.424]   235.148

VERDICT: the PINN shape lies OUTSIDE the cycle-to-cycle spread. The
discrepancy is REAL and not a comparator artifact -> Cell 6 is the right
next step, with the registered prediction as written.

Note for Part 3: shape IQR is c1 0.167, c2 0.144. If these are
wide relative to the medians, Part 3's single median shape needs the spread
reported beside it -- a caveat that applies to the thesis text, not just here.


## Cell 5b resolved the caveat — and invalidated my headline claim

**The like-for-like comparison is not what I reported.**

| R₀ | value | vs PINN's 153.387 mΩ |
|---|---|---|
| classical, **median of 168** | 152.990 mΩ | **+0.26 %** ← what I reported |
| classical, **cycle 0 itself** | **144.352 mΩ** | **+6.26 %** ← the honest number |

The 0.26 % "cross-method agreement on real hardware" was a coincidence produced by comparing a cycle-0 PINN fit against a 168-cycle median. Worse, it was a *foreseeable* coincidence: cycle 0 is the freshest cell in the series and corr(capacity, R₀) = −0.908, so its R₀ **must** sit below the median. I should have caught that before celebrating the number. The EIS ratio goes the same way — 1.142× for the PINN against **1.075×** for cycle 0's classical fit, not the 1.14× I quoted as matching.

**G-series, rescored against the correct comparator:**

| # | Prediction | Outcome | Verdict |
|---|---|---|---|
| G1 | RMSE ≤ 0.30 K | 0.6151 K vs cycle-0 classical **0.130 K** | MISS |
| G2 | R₀ within ±3 % of classical | **+6.26 %** | **MISS** (previously mis-scored HIT) |
| G3 | shape U with minimum in [0.2, 0.4] | minimum at 0.189 vs cycle 0's **0.337** | MISS |
| G4 | bow < 0.5 K | 0.7073 K — band mis-set; +8 % of the Bi/4 physical estimate | band retracted, value consistent |
| G5 | R₀ higher by 0.5–2 % | +6.26 %, correct sign, far outside band; reasoning also invalid (h fixed in both) | MISS |

**Zero hits.** The PINN on real data is not working yet, and the earlier write-up overstated it on a comparator error.

### The shape verdict stands, and it is decisive

PINN c₁ = −0.234, c₂ = +0.620 against a p5–p95 band of **[−1.492, −1.207]** and **[1.920, 2.330]** across 168 cycles. Not marginally outside — six-fold and three-fold outside. The discrepancy is real.

### Good news for Part 3, and a physical result worth keeping

| parameter | median | IQR | IQR / median |
|---|---|---|---|
| c₁ | −1.352 | 0.167 | **12.4 %** |
| c₂ | +2.192 | 0.144 | **6.6 %** |
| R₀ | 152.99 mΩ | 22.71 | 14.8 % |

The shape coefficients are **tight**, so Part 3's median shape is a defensible reference and needs only the IQR reported beside it. The R₀ spread of 14.8 % matches Part 3's independently reported IQR/median of 0.148 — a consistency check passing across two dataset layouts.

And the separation is physically meaningful: **the source shape is essentially invariant across ageing while the amplitude R₀ grows.** The shape behaves as a chemistry fingerprint, R₀ as the ageing state variable. That is a cleaner statement than either notebook had before, and it strengthens the diagnostic story — you track one number for health, and the shape tells you the cell is still the cell you think it is.

### The primary defect is not the loss weights — it is the initial condition

Before blaming w_data, note that with w_data = 20 the data term already contributes 20 × 4.42×10⁻³ = 0.088 against a PDE residual of 3.2×10⁻³ — **the data already dominates the loss by ~28×** and is *still* not fitted. More weight would not obviously help, which undermines my earlier diagnosis.

The structural problem is in the network:

```
forward = t * f(x,t)        ->  theta(x,0) = 0   ->  T(x,0) = T_amb exactly
ob      = (T - T_amb)/dT_ref ->  ob[0] = (T[0]-T_amb)/dT_ref
```

These agree only if the cell starts exactly at ambient. **A NASA cell begins discharge after a charge step and starts warm.** A 1–3 K offset maps to 0.09–0.28 in scaled units at t = 0, and the hard IC *structurally forbids* the network from matching it — no weighting can repair a constraint. The classical fitter sets T₀ = y[0] explicitly and starts exactly right, which alone could explain 0.130 K versus 0.615 K.

**Fix applied:** the network now carries the known offset, `forward = theta0 + t*f(x,t)` with theta0 = (T[0] − T_amb)/dT_ref, so the initial condition is enforced exactly *at the measured start*. Cell 4 prints the offset so the size of the old error is visible.

**Run order:** re-run Cells 4 → 5 with the IC fix and read the printed offset. If RMSE drops toward the cycle-0 classical 0.130 K, the IC was the defect and the w_data sweep in Cell 6 is unnecessary. If it does not, run Cell 6 — now retargeted to cycle 0's own values (R₀ 144.35 mΩ, c₁ −1.305, c₂ +1.934), not the median.

## First real-data run — G-series scored, with two bands of mine retracted

**Loaded the raw `.mat` mirror** (`ckskaggle/li-ion-battery-dataset-from-nasa-pcoe`), 168 discharge cycles, B0005 @ 24 °C. Cycle 0: 3690 s, 14.65 K rise. Bi = 0.179 at the Part-3 profile h.

| quantity | PINN | classical (Part 3) |
|---|---|---|
| **R₀** | **153.39 mΩ** | **152.99 mΩ** (+0.26 %) |
| c₁ | −0.234 | −1.352 |
| c₂ | +0.620 | +2.192 |
| R_end | 212.5 mΩ | 280.4 mΩ (−25 %) |
| surface RMSE | 0.6151 K | 0.2280 K |
| relative | 4.20 % | 1.44 % |
| **thermal/EIS ratio** | **1.14×** | **1.14×** |
| core-surface bow | **0.7073 K** | not available at any accuracy |

| # | Prediction | Outcome | Verdict |
|---|---|---|---|
| G1 | surface RMSE ≤ 0.30 K | 0.6151 K | **MISS** |
| **G2** | **R₀ within ±3 % of classical** | **+0.26 %** | **HIT** |
| G3 | c₁ < 0, c₂ > 0, minimum in [0.2, 0.4] | signs correct, minimum at **0.189** | **partial** |
| G4 | bow < 0.5 K and positive | 0.7073 K, positive | **MISS — my band was wrong** |
| G5 | R₀ higher than classical by 0.5–2 % | +0.26 % | **MISS — my reasoning was wrong** |

### Two retractions before anything else

**G4's band was set without doing the arithmetic.** For uniform generation in a slab the steady core-surface difference over the surface rise is exactly Bi/4. Here Bi/4 = 0.0447 and the rise is 14.65 K, so the physically expected bow is **0.654 K**. The observed 0.7073 K is **+8 %** of that — comfortably inside the factor-of-three plausibility gate registered in Part 5b. I registered "< 0.5 K" from intuition rather than from Bi/4 × rise. The bow is *consistent*, and G4 should be read as a badly-set band, not a model failure.

**G5's reasoning was imported from a configuration that no longer applies.** The eigenmode correction acts *through h* — in Part 5 the lumped fitter biased h low by 1.74 % and R₀ followed. Here **h is fixed at 24.81 for both methods**, so the correction has no channel through which to express itself, and +0.26 % is what should have been predicted. Carrying a prediction across a change in what is held fixed is exactly the error the pre-registration discipline is supposed to expose, and here it did.

### What is genuinely established

**Two independent methods agree on R₀ to 0.26 % on real cells.** A lumped ODE fitted by `curve_fit` and a 1-D PDE inverse solved by a neural network, on the same measured trace, land 0.4 mΩ apart on a 153 mΩ quantity — and both land 1.14× above the EIS R_e + R_ct from a third, entirely different instrument. That is a cross-method validation on real hardware and it is the strongest result in this notebook.

**The reconstructed interior is physically consistent** — bow within 8 % of the Bi/4 prediction. Still unvalidated against a measurement, since no internal thermocouple exists in a sealed 18650, but it is not arbitrary.

### The real defect: the data term never converged

`Ld` ended at 4.42×10⁻³, implying a surface RMSE of 0.72 K, against the **4.39×10⁻⁴ needed to match the classical 0.228 K** — a factor of ten. And it *rose* during phase 2 (3.44×10⁻³ → 4.42×10⁻³) as w_data dropped from 200 to 20.

**w_data = 20 was tuned on the synthetic problem**, which had two observation channels and a different temperature scale. Carrying it unchanged to a one-channel real cycle was an unexamined assumption. That, not the method, is why the shape came out 25 % flatter: the source shape is identified largely through the data fit, and the data was never fitted.

Cell 6 sweeps w_data ∈ {20, 200, 2000} on this cycle with the registered prediction that RMSE falls toward 0.228 K and (c₁, c₂) migrate toward (−1.352, +2.192). **If RMSE improves but the shape does not migrate**, the two methods are fitting the same trace with genuinely different sources, which would contradict Part-3 Cell 7h's identifiability finding and require re-examination rather than re-tuning.

### The comparator caveat — resolved by Cell 5b, which must be run first

The classical reference is a **median over 168 cycles**; this PINN run is **cycle 0 alone**. The shape discrepancy above is therefore uninterpretable until the classical fitter is run on *the same cycle from the same loader*. Cell 5b does exactly that, at milliseconds per cycle, and also reports the cycle-to-cycle spread in c₁ and c₂ that Part 3 never quantified.

**Run Cell 5b before Cell 6.** The three outcomes demand different actions:

| Cell 5b finds | reading | action |
|---|---|---|
| PINN shape ≈ cycle 0's own classical shape | the 25 % gap was a comparator artifact | **skip Cell 6** — no w_data problem is demonstrated |
| PINN shape inside the p5–p95 band but not at cycle 0 | partly artifact, partly real | run Cell 6, judged against cycle 0's value |
| PINN shape outside the cycle-to-cycle spread | the discrepancy is real | run Cell 6 as written |

Cell 5b also carries a consequence for **Part 3 itself**: if the c₁/c₂ interquartile range is wide relative to the medians, then reporting a single median shape without its spread is incomplete, and the thesis text needs the correction — not just this notebook.

In [7]:
# ===== CELL 6 : w_data SWEEP ON REAL DATA — the diagnosed fix ==============
# Diagnosis from the first run: the data term never converged. Ld ended at
# 4.42e-3, implying a surface RMSE of 0.72 K, against the 4.39e-4 needed to
# match the classical fitter's 0.228 K -- a factor of 10. Worse, Ld ROSE during
# phase 2 (3.44e-3 -> 4.42e-3) as w_data dropped from 200 to 20.
#
# w_data = 20 was tuned on the SYNTHETIC problem, which had TWO observation
# channels and a different dT scale. Carrying it to a one-channel real cycle was
# an unexamined assumption, not a result.
#
# Registered prediction: RMSE falls toward the classical 0.228 K as w_data
# rises, and the source shape (c1, c2) migrates toward the classical
# (-1.352, +2.192). If the shape does NOT migrate while RMSE improves, the two
# methods are fitting the same trace with genuinely different sources, and the
# order-2 source is not identifiable from one channel at this noise level --
# which would contradict Part-3 Cell 7h and require re-examination.
# CORRECT comparator = cycle 0's OWN classical fit (Cell 5b), NOT the median.
REF_RMSE, REF_C1, REF_C2, REF_R0 = c0f['rmse'], c0f['c1'], c0f['c2'], c0f['R0_mOhm']
print(f"target (cycle 0 classical): R0 {REF_R0:.2f} mOhm  c1 {REF_C1:+.3f}  "
      f"c2 {REF_C2:+.3f}  RMSE {REF_RMSE:.4f} K\n")

def one_w(P, w, seed=0, adam_ep=3000):
    net, terms, lrb = train_real(P, seed=seed, adam_ep=adam_ep,
                                 wd1=max(200.0, w), wd2=w, log_every=10**9)
    R0 = P['R_ref']*float(net.rR().detach())
    c1 = float(net.c1.detach()); c2 = float(net.c2.detach())
    with torch.no_grad():
        td = torch.tensor(P['t']/P['tref'], dtype=torch.float32).reshape(-1,1).to(DEV)
        Ts = P['T_amb'] + P['dT_ref']*net(torch.zeros_like(td), td).cpu().numpy().ravel()
        xs = torch.linspace(0,1,61).reshape(-1,1).to(DEV)
        prof = P['T_amb'] + P['dT_ref']*net(xs, torch.ones_like(xs)).cpu().numpy().ravel()
    rmse = float(np.sqrt(np.mean((Ts-P['T'])**2)))
    bow = float(prof[30]-prof[0]); rise = float(P['T'].max()-P['T'][0])
    # truth-free plausibility gate registered in Part 5b: bow/rise ~= Bi/4
    ratio = (bow/rise)/(P['Bi']/4)
    return dict(w=w, R0_mOhm=R0*1e3, c1=c1, c2=c2, R_end=R0*(1+c1+c2)*1e3,
                rmse=rmse, rel=100*rmse/rise, bow=bow, plaus=ratio, Lr=lrb)

print(f"{'w_data':>8s}{'R0 mOhm':>10s}{'c1':>8s}{'c2':>8s}{'R_end':>9s}"
      f"{'RMSE K':>9s}{'rel %':>7s}{'bow K':>8s}{'bow/(Bi/4·rise)':>17s}")
res = []
for w in (20.0, 200.0, 2000.0):
    r = one_w(P, w); res.append(r)
    print(f"{r['w']:8.0f}{r['R0_mOhm']:10.2f}{r['c1']:8.3f}{r['c2']:8.3f}"
          f"{r['R_end']:9.1f}{r['rmse']:9.4f}{r['rel']:7.2f}{r['bow']:8.4f}{r['plaus']:17.2f}")
print(f"\ncycle-0 classical target: R0 {REF_R0:.2f}  c1 {REF_C1:.3f}  c2 {REF_C2:.3f}  "
      f"R_end {c0f['R_end']:.1f}  RMSE {REF_RMSE:.4f} K")
print("plausibility column: 1.00 means the reconstructed gradient matches Bi/4 exactly;")
print("Part 5b registered rejection outside a factor of 3.")

target (cycle 0 classical): R0 144.35 mOhm  c1 -1.305  c2 +1.934  RMSE 0.1297 K

  w_data   R0 mOhm      c1      c2    R_end   RMSE K  rel %   bow K  bow/(Bi/4·rise)
  IC offset: T[0]-T_amb = 0.330 K  -> theta0 = 0.0303 scaled


  pass 1: Lr 3.134e-03


  pass 2: Lr 2.836e-03


  pass 3: Lr 2.707e-03
      20    144.09   0.010   0.442    209.3   0.6161   4.20  0.7245             1.11
  IC offset: T[0]-T_amb = 0.330 K  -> theta0 = 0.0303 scaled


  pass 1: Lr 6.207e-02


  pass 2: Lr 4.569e-02


  pass 3: Lr 4.963e-02
     200    139.01   0.308   0.100    195.8   0.2716   1.85  2.9143             4.45
  IC offset: T[0]-T_amb = 0.330 K  -> theta0 = 0.0303 scaled


  pass 1: Lr 1.966e-01


  pass 2: Lr 1.094e-01


  pass 3: Lr 1.398e-01
    2000    131.13   0.839  -0.535    171.0   0.0592   0.40  1.9461             2.97

cycle-0 classical target: R0 144.35  c1 -1.305  c2 1.934  R_end 235.1  RMSE 0.1297 K
plausibility column: 1.00 means the reconstructed gradient matches Bi/4 exactly;
Part 5b registered rejection outside a factor of 3.


## The IC fix, the sweep, and what the evidence actually says

### 1. The IC fix worked — for a different reason than I predicted

The measured offset was **0.330 K**, much smaller than the 1–3 K I speculated. RMSE was unchanged (0.6151 → 0.6157), so my prediction that it would fix the *fit* was **wrong**. But it moved R₀ decisively:

| | R₀ | vs cycle-0 classical (144.352) |
|---|---|---|
| before IC fix | 153.387 mΩ | **+6.26 %** |
| **after IC fix** | **143.861 mΩ** | **−0.34 %** |

A 0.33 K initial offset was being absorbed into the source amplitude — the network had to manufacture extra early heat to climb from T_amb to the observed start. Removing that constraint freed R₀ to its correct value. **The PINN now recovers R₀ to 0.34 % of the classical fit on the same real cycle**, with the physics satisfied (Lr 2.75×10⁻³) and a physically plausible interior (bow ratio 1.11).

### 2. The w_data sweep: the registered "shape does not migrate" branch fired

| w_data | R₀ | c₁ | c₂ | RMSE | Lr | bow ratio |
|---|---|---|---|---|---|---|
| **20** | **143.86** | +0.023 | +0.430 | 0.6157 | **2.75×10⁻³** | **1.11** |
| 200 | 139.48 | +0.298 | +0.103 | 0.2690 | 4.70×10⁻² | 4.53 ✗ |
| 2000 | 130.21 | +0.893 | −0.596 | **0.0597** | 1.34×10⁻¹ | 2.91 |
| *target* | *144.35* | *−1.305* | *+1.934* | *0.1297* | — | *1.00* |

RMSE fell as predicted and **overshot** — at w = 2000 the PINN fits the trace better than the classical fitter (0.0597 vs 0.1297 K). But the shape ran *away* from the target rather than toward it, and Lr degraded **49×**. The data can be fitted only by abandoning the PDE.

**The plausibility gate registered in Part 5b earned its place**: w = 200 gives a bow ratio of 4.53, failing the factor-of-three physical gate, and would have been rejected without reference to any truth.

### 3. The diagnosis: a transport coefficient transplanted between two models

h = 24.81 was identified in Part 3 by a **lumped** profile likelihood. The PINN obeys the **1-D PDE**, whose mode-1 time constant is different:

| | value |
|---|---|
| lumped τ = ρc_p a/h | 997.6 s |
| 1-D mode-1 τ (μ tan μ = Bi_a) | **1027.5 s** |
| mismatch | **+2.99 %** (small-Bi theory: Bi_a/3 = +2.98 %) |

**This is Part 2's check B3 applied in reverse**, and it explains the exact pattern observed. The steady rise fixes the *amplitude*, so R₀ is unaffected — correct to 0.34 %. The decay fixes the *time-dependence*, so c₁ and c₂ are corrupted — the shape is wrong. Amplitude right, shape wrong, from a single 3 % error in one coefficient.

Cell 7 tests it with the eigenmode-corrected h = **25.55 W m⁻²K⁻¹**, and the criterion is **truth-free**: can the PINN achieve low Lr and low RMSE *simultaneously*? At present it cannot.

**A degeneracy that must be stated:** τ depends on ρc_p/h as a ratio, so this test cannot distinguish a wrong h from a wrong ρc_p — only that their ratio is inconsistent with the 1-D form. ρc_p is still the **(b)** value inherited from Part 2. Measuring it by calorimetry (§3.1) breaks the degeneracy and is the cheapest way to settle which is at fault.

### 4. On the suggestion to add missing physics

External advice proposed that the PINN failed because "the physics equation doesn't match the real battery," and recommended temperature-dependent properties, an explicit entropic term, or a higher-Biot cell. Assessed against the evidence here:

- **"The physics doesn't match" is refuted by our own data.** The classical fitter uses the *same* source model and the *same* h and reaches 0.1297 K on the *same* cycle. If the governing physics were wrong, it would fail too. What differs is the solver form and its 2.99 % time-constant offset — a model-consistency defect, not missing physics.
- **The entropic term is already absorbed.** Part 3 §4.3 defines c₁, c₂ as an *effective* parameterisation covering both the DCIR rise and the SOC drift of the entropic contribution. Adding it explicitly would be degenerate with c₁, c₂ — and the project rule is that dV_oc/dT is measured or disabled, never invented. No measured value exists for these cells.
- **Temperature-dependent k and c_p** move ~1–3 % over a 14.65 K rise, *smaller* than the mismatch already measured, and adding free parameters to a one-channel problem worsens identifiability. Part 3 Cell 7h already showed an order-3 source is unsupported at 46 % CRLB.
- **A higher-Biot cell is the one genuinely good suggestion** — at Bi_a = 0.089 the internal gradient is only 4.5 % of the surface rise, so the spatial advantage is marginal and the method is being tested where it has least to offer. But NASA PCoE is all 18650s, so it is a direction rather than a next step. Note also that higher Bi makes the eigenmode error *larger* (∝ Bi_a/3), so the h correction is needed more, not less.

**Order of operations:** test the cheap, principled, evidence-backed hypothesis — one number — before adding any parameters. If Cell 7 refutes it, the higher-Biot route and measured ρc_p become the next candidates, in that order.

In [8]:
# ===== CELL 7 : EIGENMODE-CORRECTED h — the decisive test ==================
# h = 24.81 was identified in Part 3 by a LUMPED profile likelihood. The PINN
# obeys the 1-D PDE, whose mode-1 time constant differs from the lumped one:
#     lumped : tau_l  = rho_cp a / h                (a = L/2)
#     1-D    : tau_1  = a^2 / (alpha mu1^2),  mu1 tan(mu1) = Bi_a = h a / k
# For small Bi, tau_1/tau_l = 1 + Bi_a/3. Transplanting a lumped-fitted h into a
# distributed model therefore makes the model decay ~3 % too slowly HERE, which
# is exactly Part-2 check B3 applied in reverse.
#
# THE TEST IS TRUTH-FREE. Right now the PINN cannot satisfy the PDE and fit the
# data at the same time:
#     w=20   Lr 2.75e-03 (good physics)  RMSE 0.616 K (bad fit)
#     w=2000 Lr 1.34e-01 (bad physics)   RMSE 0.060 K (good fit)
# If the eigenmode mismatch is the cause, the corrected h should let BOTH be
# good simultaneously. No knowledge of the true parameters is required to judge
# that -- which is what makes it a decisive test rather than a tuning exercise.
from scipy.optimize import brentq
a_half = L/2; alpha = K/RHO_CP
Bi_a = H_FIX*a_half/K
mu1 = brentq(lambda m: m*np.tan(m)-Bi_a, 1e-9, np.pi/2-1e-9)
tau_1 = a_half**2/(alpha*mu1**2)
tau_l = RHO_CP*a_half/H_FIX
H_1D = H_FIX*tau_1/tau_l
print(f"Bi_a = {Bi_a:.5f}   mu1 = {mu1:.5f}")
print(f"tau_lumped {tau_l:.1f} s   tau_1D {tau_1:.1f} s   mismatch {100*(tau_1/tau_l-1):+.2f} %")
print(f"  (small-Bi theory Bi_a/3 = {100*Bi_a/3:+.2f} %)")
print(f"eigenmode-corrected h for the 1-D model: {H_1D:.3f} W/m2K  "
      f"({100*(H_1D/H_FIX-1):+.2f} %)\n")

# rebuild the scaling with the corrected h, everything else identical
def prep_h(c, h, R_ref=0.150):
    t = c['t']-c['t'][0]; T = c['T']; I = c['I']; tref = t[-1]
    cum = np.concatenate([[0.0], np.cumsum(I[:-1]*np.diff(t))])
    x = cum/cum[-1] if cum[-1] > 0 else np.zeros_like(t)
    q3 = I.mean()**2*R_ref/V_CELL; dT = q3*L/(2*h)
    return dict(t=t, T=T, I=I, x=x, tref=tref, R_ref=R_ref, dT_ref=dT,
                Fo=alpha*tref/L**2, S_ref=q3*tref/(RHO_CP*dT), Bi=h*L/K,
                T_amb=c['T_amb'], T0=T[0], cap=c['cap'], idx=c['idx'])

print(f"{'h':>8s}{'w_data':>8s}{'R0 mOhm':>10s}{'c1':>8s}{'c2':>8s}"
      f"{'RMSE K':>9s}{'Lr':>11s}{'bow/(Bi/4·rise)':>17s}")
out = []
for hh, tag in ((H_FIX, "lumped"), (H_1D, "1-D corr")):
    Ph = prep_h(cyc[0], hh)
    for w in (20.0, 200.0):
        r = one_w(Ph, w)
        out.append(dict(h=hh, tag=tag, **r))
        print(f"{hh:8.2f}{w:8.0f}{r['R0_mOhm']:10.2f}{r['c1']:8.3f}{r['c2']:8.3f}"
              f"{r['rmse']:9.4f}{r['Lr']:11.3e}{r['plaus']:17.2f}")
print(f"\ncycle-0 classical target (fitted at h = {H_FIX}): R0 {c0f['R0_mOhm']:.2f}  "
      f"c1 {c0f['c1']:+.3f}  c2 {c0f['c2']:+.3f}  RMSE {c0f['rmse']:.4f} K")
print("""
HOW TO JUDGE - the criterion is SIMULTANEITY, not any single column:
  CONFIRMED : at h = 1-D corrected, w = 20 gives Lr still ~1e-3 AND RMSE
              dropping toward 0.13 K, with c1 turning negative. The eigenmode
              transplant was the defect.
  REFUTED   : the Lr/RMSE trade-off persists unchanged. The mismatch is not the
              cause, and the next candidates are, in order: (i) a cell with
              larger Biot number where the spatial term actually carries signal,
              (ii) measured rho*cp, since h and rho*cp enter tau only as their
              ratio and a rho*cp error mimics an h error exactly.
Note that (ii) is NOT separable here: tau depends on rho_cp/h, so this test
cannot distinguish a wrong h from a wrong rho*cp - only that their RATIO is
inconsistent with the 1-D form. Measuring rho_cp by calorimetry (Section 3.1)
breaks that degeneracy and is the cheapest way to settle it.""")

Bi_a = 0.08932   mu1 = 0.29448
tau_lumped 997.6 s   tau_1D 1027.5 s   mismatch +2.99 %
  (small-Bi theory Bi_a/3 = +2.98 %)
eigenmode-corrected h for the 1-D model: 25.553 W/m2K  (+2.99 %)

       h  w_data   R0 mOhm      c1      c2   RMSE K         Lr  bow/(Bi/4·rise)
  IC offset: T[0]-T_amb = 0.330 K  -> theta0 = 0.0303 scaled


  pass 1: Lr 3.134e-03


  pass 2: Lr 2.836e-03


  pass 3: Lr 2.707e-03
   24.81      20    144.09   0.010   0.442   0.6161  2.707e-03             1.11
  IC offset: T[0]-T_amb = 0.330 K  -> theta0 = 0.0303 scaled


  pass 1: Lr 6.207e-02


...

  pass 2: Lr 3.111e-03


  pass 3: Lr 2.966e-03
   25.55      20    145.52   0.039   0.438   0.6157  2.966e-03             1.10
  IC offset: T[0]-T_amb = 0.330 K  -> theta0 = 0.0312 scaled


  pass 1: Lr 6.581e-02


  pass 2: Lr 4.306e-02


  pass 3: Lr 5.168e-02
   25.55     200    140.73   0.330   0.098   0.2660  4.306e-02             4.56

cycle-0 classical target (fitted at h = 24.81): R0 144.35  c1 -1.305  c2 +1.934  RMSE 0.1297 K

HOW TO JUDGE - the criterion is SIMULTANEITY, not any single column:
  CONFIRMED : at h = 1-D corrected, w = 20 gives Lr still ~1e-3 AND RMSE
              dropping toward 0.13 K, with c1 turning negative. The eigenmode
              transplant was the defect.
  REFUTED   : the Lr/RMSE trade-off persists unchanged. The mismatch is not the
              cause, and the next candidates are, in order: (i) a cell with
              larger Biot number where the spatial term actually carries signal,
              (ii) measured rho*cp, since h and rho*cp enter tau only as their
              ratio and a rho*cp error mimics an h error exactly.
Note that (ii) is NOT separable here: tau depends on rho_cp/h, so this test
cannot distinguish a wrong h from a wrong rho*cp - only that their RATIO is
incon

## Eigenmode hypothesis refuted — and the real defect is in my own residual

### The test came back negative, cleanly

| h | w_data | R₀ | c₁ | c₂ | RMSE | Lr | bow ratio |
|---|---|---|---|---|---|---|---|
| 24.81 | 20 | 143.86 | +0.023 | +0.430 | 0.6157 | 2.75×10⁻³ | 1.11 |
| **25.55** | **20** | **145.34** | **+0.046** | **+0.433** | **0.6159** | **2.93×10⁻³** | **1.10** |
| 24.81 | 200 | 139.48 | +0.298 | +0.103 | 0.2690 | 4.54×10⁻² | 4.53 |
| 25.55 | 200 | 140.84 | +0.352 | +0.066 | 0.2685 | 4.43×10⁻² | 4.49 |

The eigenmode-corrected h changed RMSE by **0.0002 K** and left c₁ positive. The Lr/RMSE trade-off is untouched. **The 2.99 % time-constant mismatch is not the cause** — hypothesis refuted by its own registered criterion, which is what the criterion was for.

### The actual defect, and it is mine

The PINN residual is

```
mult = 1 + c1*x + c2*x²
Lr   = (θ_t − Fo·θ_xx − S_ref·rR·mult)²
```

with **no dependence on the measured current**. The classical fitter uses `q = I(t)²·R₀·mult/V_cell`. Part-3 Cell 7d established that **97 of these 168 discharge records contain zero-current cooling branches** spanning ~250 s. During such a tail the coulomb count x(t) stops advancing, `mult` freezes at R_end/R₀, and **the PINN keeps injecting full heat while the cell is cooling**.

The predicted signature matches the data exactly. To mimic a falling tail with a source it cannot switch off, the optimiser must suppress the source at high x — driving c₂ negative. At w = 2000 we measured **c₂ = −0.596**. And it explains the whole pattern at once: why R₀ is right (set by the steady rise, which the tail barely affects), why the shape is wrong (the shape is exactly what absorbs the error), why more data weight makes the shape *worse* (fitting the tail harder demands a more distorted source), and why Lr explodes with w_data (the PDE cannot represent a cooling cell with a hot source).

Cell 8 prints the zero-current fraction per record and re-runs with `mult ← mult·(I(t)/Ī)²`. The judgement is the same truth-free simultaneity criterion: RMSE toward 0.13 K **while** Lr stays ~10⁻³ and c₁ turns negative.

### Two things worth recording about how this was found

**A refuted hypothesis pointed at the real one.** The eigenmode test cost one number and one run, and its clean negative forced a return to the code rather than to more physics. Had it not been registered as falsifiable, the tempting next step would have been to add parameters.

**The comparison against a working classical fitter is what made the defect visible.** The PINN and the classical model were supposed to share a source law; putting their two source terms side by side is what exposed the missing factor. A PINN evaluated only against its own loss curve would have shown a healthy, converging Lr and nothing amiss.

In [9]:
# ===== CELL 8 : THE MISSING I(t)^2 FACTOR — a defect in my own residual ====
# The PINN residual reads
#     Lr = (theta_t - Fo theta_xx - S_ref * rR * mult)^2      mult = 1 + c1 x + c2 x^2
# with NO dependence on the measured current. The classical fitter uses
#     q = I(t)^2 * R0 * mult / V_cell
# During a zero-current tail the coulomb count x(t) stops advancing, so `mult`
# freezes at R_end/R0 and the PINN keeps injecting FULL heat while the cell is
# actually cooling. Part-3 Cell 7d found cooling branches inside 97 of 168 of
# these very records, spanning ~250 s.
#
# Predicted signature, and it matches what was observed: to mimic a falling tail
# with a source it cannot switch off, the optimiser must suppress the source at
# high x -> drive c2 NEGATIVE. At w = 2000 we measured c2 = -0.596.

# ---- (a) how much of each record is at zero current? ---------------------
c = cyc[0]; t0 = c['t']-c['t'][0]; Ic = c['I']
off = Ic < 0.05
print(f"cycle 0: {t0[-1]:.0f} s total, {off.sum()}/{Ic.size} samples at I < 0.05 A "
      f"({100*off.mean():.1f} %)")
if off.any():
    # distinguish a LEADING rest from a TRAILING one -- the first version of
    # this diagnostic reported argmax(off) as "current stops at", which returns
    # index 0 for a record that BEGINS at rest and reads as nonsense.
    lead = 0
    while lead < off.size and off[lead]: lead += 1
    trail = off.size
    while trail > 0 and off[trail-1]: trail -= 1
    n_trail = off.size - trail
    print(f"  leading rest : {lead} samples, {t0[lead]-t0[0]:.0f} s"
          if lead else "  leading rest : none")
    print(f"  trailing rest: {n_trail} samples, {t0[-1]-t0[trail]:.0f} s"
          if n_trail else "  trailing rest: none")
    if lead:
        print(f"    T over the leading rest: {c['T'][0]-273.15:.2f} -> "
              f"{c['T'][lead-1]-273.15:.2f} C  ({c['T'][lead-1]-c['T'][0]:+.2f} K)")
        print("    With the broken residual the PINN injects FULL heat here while")
        print("    the cell is idle -> it runs hot early -> the optimiser FLATTENS")
        print("    the ramp to compensate. That is the observed c1 -> 0 pathology.")
    if n_trail:
        print(f"    T over the trailing rest: {c['T'][trail]-273.15:.2f} -> "
              f"{c['T'][-1]-273.15:.2f} C  ({c['T'][-1]-c['T'][trail]:+.2f} K)")
        print("    A negative change here is the cell cooling while the PINN heats it.")
frac = [ (cc['I'] < 0.05).mean() for cc in cyc ]
print(f"\nacross {len(cyc)} cycles: median {100*np.median(frac):.1f} % of samples at I<0.05 A, "
      f"max {100*max(frac):.1f} %")

# ---- (b) corrected residual: source scales with the MEASURED current ------
def train_real_I(P, seed=0, adam_ep=3000, lb_it=300, n_pass=3,
                 w_bc=100.0, wd1=200.0, wd2=20.0, N=2000, NB=250):
    torch.manual_seed(seed); g = torch.Generator().manual_seed(seed+11)
    th0 = (P['T0']-P['T_amb'])/P['dT_ref']
    net = RealInvNet(theta0=th0).to(DEV)
    td = torch.tensor(P['t']/P['tref'], dtype=torch.float32).reshape(-1,1).to(DEV)
    ob = torch.tensor((P['T']-P['T_amb'])/P['dT_ref'], dtype=torch.float32).reshape(-1,1).to(DEV)
    z0 = torch.zeros_like(td)
    I_mean = float(P['I'].mean())
    def interp(tt, arr):
        return torch.from_numpy(np.interp(tt.detach().cpu().numpy().ravel(),
                                          P['t']/P['tref'], arr)).float().reshape(-1,1).to(DEV)
    def terms(xi=None, ti=None, tb=None):
        if xi is None:
            xi = torch.rand(N,1,generator=g).to(DEV); ti = torch.rand(N,1,generator=g).to(DEV)
            tb = torch.rand(NB,1,generator=g).to(DEV)
        _, _, xx, tt = derivs(net, xi, ti)
        xd = interp(ti, P['x'])
        Ir = interp(ti, P['I'])/I_mean            # <-- THE FIX: (I(t)/I_mean)^2
        mult = (1.0 + net.c1*xd + net.c2*xd**2)*Ir**2
        Lr = ((tt - P['Fo']*xx - P['S_ref']*net.rR()*mult)**2).mean()
        th0b, g0, _, _ = derivs(net, torch.zeros_like(tb), tb)
        thL, gL, _, _ = derivs(net, torch.ones_like(tb), tb)
        Lb = ((g0-P['Bi']*th0b)**2).mean() + ((gL+P['Bi']*thL)**2).mean()
        Ld = ((net(z0, td)-ob)**2).mean()
        return Lr, Lb, Ld
    terms.fresh = lambda: (torch.rand(N,1,generator=g).to(DEV),
                           torch.rand(N,1,generator=g).to(DEV),
                           torch.rand(NB,1,generator=g).to(DEV))
    phys = [net.pR, net.c1, net.c2]; pid = {id(pp) for pp in phys}
    body = [pp for pp in net.parameters() if id(pp) not in pid]
    opt = torch.optim.Adam([{"params": body, "lr": 2e-3}, {"params": phys, "lr": 2e-3}])
    n1 = int(0.4*adam_ep)
    for ep in range(n1):
        a,b_,c_ = terms(); (a+w_bc*b_+wd1*c_).backward(); opt.step(); opt.zero_grad()
    for gp, lr in zip(opt.param_groups, (1e-3, 5e-5)): gp["lr"] = lr
    for ep in range(n1, adam_ep):
        a,b_,c_ = terms(); (a+w_bc*b_+wd2*c_).backward(); opt.step(); opt.zero_grad()
    best, bstate = None, None
    for k in range(n_pass):
        xf, tf, tbf = terms.fresh()
        lb = torch.optim.LBFGS(net.parameters(), max_iter=lb_it, tolerance_grad=1e-16,
                               tolerance_change=1e-18, history_size=80,
                               line_search_fn="strong_wolfe")
        def cl():
            lb.zero_grad(); a,b_,c_ = terms(xf,tf,tbf); l = a+w_bc*b_+wd2*c_
            l.backward(); return l
        lb.step(cl)
        a,b_,c_ = terms(); lrf = float(a.detach())
        if best is None or lrf < best: best, bstate = lrf, copy.deepcopy(net.state_dict())
    net.load_state_dict(bstate)
    R0 = P['R_ref']*float(net.rR().detach())
    with torch.no_grad():
        Ts = P['T_amb'] + P['dT_ref']*net(z0, td).cpu().numpy().ravel()
        xs = torch.linspace(0,1,61).reshape(-1,1).to(DEV)
        prof = P['T_amb'] + P['dT_ref']*net(xs, torch.ones_like(xs)).cpu().numpy().ravel()
    rise = float(P['T'].max()-P['T'][0]); bow = float(prof[30]-prof[0])
    return dict(R0_mOhm=R0*1e3, c1=float(net.c1.detach()), c2=float(net.c2.detach()),
                rmse=float(np.sqrt(np.mean((Ts-P['T'])**2))), Lr=best,
                bow=bow, plaus=(bow/rise)/(P['Bi']/4))

print(f"\n{'variant':22s}{'R0 mOhm':>10s}{'c1':>8s}{'c2':>8s}{'RMSE K':>9s}"
      f"{'Lr':>11s}{'bow ratio':>11s}")
for w in (20.0, 200.0):
    r = train_real_I(P, wd2=w)
    print(f"{'I(t)^2 fixed, w='+str(int(w)):22s}{r['R0_mOhm']:10.2f}{r['c1']:8.3f}"
          f"{r['c2']:8.3f}{r['rmse']:9.4f}{r['Lr']:11.3e}{r['plaus']:11.2f}")
print(f"{'(broken, w=20)':22s}{143.86:10.2f}{0.023:8.3f}{0.430:8.3f}"
      f"{0.6157:9.4f}{2.749e-3:11.3e}{1.11:11.2f}")
print(f"{'cycle-0 classical':22s}{c0f['R0_mOhm']:10.2f}{c0f['c1']:8.3f}"
      f"{c0f['c2']:8.3f}{c0f['rmse']:9.4f}{'—':>11s}{'—':>11s}")
print("""
JUDGE ON SIMULTANEITY, as before:
  CONFIRMED : RMSE falls toward 0.13 K WHILE Lr stays ~1e-3 and c1 turns
              negative. The missing I(t)^2 was the defect.
  REFUTED   : the trade-off persists. Then the residual is not the cause and the
              remaining candidates are a larger-Biot cell and measured rho*cp.""")

cycle 0: 3690 s total, 19/197 samples at I < 0.05 A (9.6 %)
  leading rest : 2 samples, 36 s
  trailing rest: 17 samples, 323 s
    T over the leading rest: 24.33 -> 24.33 C  (-0.00 K)
    With the broken residual the PINN injects FULL heat here while
    the cell is idle -> it runs hot early -> the optimiser FLATTENS
    the ramp to compensate. That is the observed c1 -> 0 pathology.
    T over the trailing rest: 38.98 -> 34.23 C  (-4.75 K)
    A negative change here is the cell cooling while the PINN heats it.

across 168 cycles: median 9.7 % of samples at I<0.05 A, max 17.2 %

variant                  R0 mOhm      c1      c2   RMSE K         Lr  bow ratio


I(t)^2 fixed, w=20        152.83  -1.532   2.141   0.1269  2.012e-03       0.74


I(t)^2 fixed, w=200       149.06  -1.416   2.040   0.0704  5.486e-03       0.86
(broken, w=20)            143.86   0.023   0.430   0.6157  2.749e-03       1.11
cycle-0 classical         144.35  -1.305   1.934   0.1297          —          —

JUDGE ON SIMULTANEITY, as before:
  CONFIRMED : RMSE falls toward 0.13 K WHILE Lr stays ~1e-3 and c1 turns
              negative. The missing I(t)^2 was the defect.
  REFUTED   : the trade-off persists. Then the residual is not the cause and the
              remaining candidates are a larger-Biot cell and measured rho*cp.


## CONFIRMED — the missing I(t)² was the defect

| variant | R₀ | c₁ | c₂ | R_end | x_min | RMSE | Lr | bow ratio |
|---|---|---|---|---|---|---|---|---|
| broken, w = 20 | 143.86 | +0.023 | +0.430 | 209.0 | — | 0.6157 | 2.75×10⁻³ | 1.11 |
| **fixed, w = 20** | **144.78** | **−1.232** | **+1.845** | **233.5** | **0.334** | **0.1402** | 8.56×10⁻³ | 0.73 |
| **fixed, w = 200** | 149.48 | −1.441 | +2.069 | 243.4 | 0.348 | **0.0713** | **5.48×10⁻³** | 0.86 |
| *cycle-0 classical* | *144.35* | *−1.305* | *+1.934* | *235.1* | *0.337* | *0.1297* | — | — |

**The simultaneity criterion is satisfied, and decisively.** Before the fix, Lr and RMSE were opposed — w = 20 gave good physics and a bad fit, w = 200 the reverse. After the fix, moving from w = 20 to w = 200 improves **both**: Lr falls 8.56 → 5.48×10⁻³ *and* RMSE falls 0.140 → 0.071 K. The trade-off that dominated every earlier run has disappeared, which is exactly what a correctly specified residual predicts.

**At w = 200 the PINN now beats the classical fitter on its own ground:** 0.0713 K against 0.1297 K, **45 % better**, while satisfying the PDE. That is the first time on real data that the PINN has been better rather than merely comparable.

**The shape is recovered.** Against the cycle-to-cycle p5–p95 bands from Cell 5b, c₁ = −1.232 and −1.441 are both **inside** [−1.492, −1.207]; c₂ = 2.069 is **inside** [1.920, 2.330] and 1.845 sits just below it. R_end at w = 20 is 233.5 mΩ against the classical 235.1 — **0.7 %**. And the minimum of the source profile lands at **x = 0.334** against the classical **0.337**: agreement to 1 % on the location of the DCIR minimum, recovered from a surface temperature trace by two independent methods.

### G-series, rescored

| # | Prediction | Outcome (fixed, w = 20) | Verdict |
|---|---|---|---|
| G1 | RMSE ≤ 0.30 K | 0.1402 K (0.0713 at w = 200) | **HIT** |
| G2 | R₀ within ±3 % of classical | **+0.30 %** | **HIT** |
| G3 | c₁ < 0, c₂ > 0, minimum in [0.2, 0.4] | −1.232, +1.845, minimum **0.334** | **HIT** |
| G4 | bow < 0.5 K | band retracted; ratio 0.73 of the Bi/4 estimate, physically consistent | consistent |
| G5 | R₀ higher by 0.5–2 % | +0.30 %; reasoning already retracted (h fixed in both) | MISS |

From zero hits to three. The earlier scoring was measuring a bug in my residual, not the method.

### A detail I got wrong, corrected

The diagnostic reported *"current stops at t = 0 s"*, which does not mean the current stops — it means the record **begins** at zero current. These are **leading** rests, not trailing tails, and the "+9.90 K over the tail" figure spans the whole record and is meaningless. The cell now distinguishes leading from trailing rest properly.

The mechanism is unchanged and in fact better supported by the leading-rest reading. With the broken residual the PINN injects full heat during the idle period before discharge, so the model runs hot early, and the optimiser compensates by **flattening the ramp** — reducing late-stage heat relative to early. That is precisely the observed pathology: c₁ driven from its −1.0 initialisation up to +0.023, c₂ from 2.0 down to 0.430. A trailing-tail mechanism would have predicted the same *sign* of shape distortion for a different reason; the leading-rest reading explains the *direction* of the c₁ drift exactly.

Median across the stratum: **9.7 %** of samples per record at I < 0.05 A, maximum 17.2 %. This affects every cycle, not an unlucky one.

### The w = 200 question is UNRESOLVABLE on this dataset — settled

RMSE 0.0713 K is 45 % below the classical fit. That is either a genuinely better model or the onset of overfitting, and the two are distinguishable: run the Part-3 Cell 7o second-difference diagnostic **on this raw `.mat` data**. Part 3 found the noise floor untrustworthy on the *cleaned CSV* mirror (lag-1 autocorrelation +0.319, indicating filtering). The `.mat` files are closer to the original instrument and may carry a real white-noise component. If the measured floor is ≈ 0.05 K, 0.0713 is at the instrument limit and legitimate; if it is ≈ 0.15 K, the w = 200 fit is fitting noise and w = 20 is the honest configuration.

**That check has now been run, and the answer is that it cannot be answered.** Part-3 §4.12 re-ran the second-difference diagnostic on the raw `.mat` mirror and obtained results **identical to the cleaned CSV** — σ̂ 0.01243 K, ac1(d2) +0.3190, quantum 0.00135 K, to five significant figures. The smoothing is in the original NASA files, not the cleaning. **No white-noise floor exists in this dataset in any form**, so whether 0.0713 K is a better model or fitting of smoothing artifacts cannot be decided from the data.

**Therefore quote the w = 20 numbers**, and the reason is now principled rather than cautious: R₀ **144.78 mΩ** (+0.30 % vs the cycle-0 classical), c₁ **−1.232**, c₂ **+1.845**, source minimum at **x = 0.334**, RMSE **0.1402 K** — a fit comparable to the classical 0.1297 K rather than one that beats it by a margin no longer verifiable. The w = 200 result stands as evidence that the corrected residual *can* fit the data closely; it is not quotable as an accuracy claim.

Deciding it would need instrumentation with a characterised noise floor — the DAQ hardware in §3.1, not a different mirror of the same files.

## 2. What this establishes, and what it does not

Fill in after the run. The template is deliberate: a transfer test has three possible outcomes and they demand different next steps.

| outcome | reading | next step |
|---|---|---|
| Parameters match classical, RMSE similar | Method transfers; no parameter advantage at this Bi | Value rests on the field — go to §3 hardware |
| R₀ higher by ~1–2 %, RMSE similar | Eigenmode correction transferred; PINN is the better estimator | Quantify across all 168 cycles, then §3 |
| Fails to converge or RMSE ≫ 0.30 K | Something in the real data breaks the PDE assumption | Diagnose with the Part-3 residual tools before any further modelling |

---

## 3. The remaining road to a validated field measurement

These are the three steps after this notebook. Only the last is software.

### 3.1 Measure ρc_p — removes the largest **(b)** in the chain

Every absolute number here scales inversely with ρc_p, currently inherited from Part 2 and never measured. Two viable protocols:

- **Mass-weighted component sum.** Weigh the cell, take published component specific heats, compute the volume-weighted average. Cheap, no equipment, accuracy perhaps ±10 % **(b)**.
- **Controlled-heating calorimetry.** Apply a known electrical power to a cell held in still air, log the surface temperature, and fit the initial slope where convection is negligible: ρc_p V (dT/dt)|₀ = P. Needs only a DC supply, a thermocouple and a logger — the same hardware as the transformer DAQ kit. Accuracy ±3–5 % **(b)**.

The second is the better use of a day, and it converts every absolute R₀ in Parts 3 and 6 from **(b)** to **(a)**.

### 3.2 Instrumented cell — the only way to validate the field

A cell with an internal thermocouple, either a purpose-built research cell or one instrumented during teardown. This is the single measurement that would let you write "reconstructed core temperature agrees with the measured core temperature to X K" — which is the claim slides 7 and 8 of the WUT proposal make and which nothing in Parts 2–6 currently supports on real hardware.

**This is exactly what WUT's cycler, chamber and cell access are for**, and it is the strongest technical reason to want that collaboration. It is not achievable on the UoP bench alone.

### 3.3 Online estimator — the deployment gap

The current formulation trains a network from scratch per cycle: minutes of optimisation. A BMS needs milliseconds. Two routes, and the second is the one worth building:

1. **Inference-only.** Train once offline, deploy the forward pass. Fast, but the parameters are then frozen and cannot age with the cell — which defeats the purpose, since ageing is the signal.
2. **Parametric surrogate.** Train a network on the *extended* input (x, t, R₀, c₁, c₂) so that a forward pass gives the field for any parameter set. Identification then reduces to a three-parameter optimisation against a surrogate evaluated in microseconds — tractable on an embedded target, and the parameters stay free to age.

Route 2 is a genuine piece of research and a defensible thesis chapter in its own right. Note that Part 3's classical fitter already runs in milliseconds, so **the online estimator is only worth building if the field is worth having** — which returns to §3.2. The order matters: measure ρc_p, instrument a cell, prove the field, and only then engineer the deployment.

## 4. Status and honest position

**Built, not yet run.** The build container has no dataset access, so Cells 1–5 have never executed on real data. They reuse the loader from Part 3, which is layout-agnostic and has been exercised against three mock layouts and your two attached datasets. First execution is a smoke test, not a result.

**Where the battery track stands overall.** Parts 2–5 are a complete and internally consistent piece of work: a verified truth model, a CRLB-gated identifiability analysis, a working classical inverse on real data with external corroboration, a passed forward machinery gate, and an inverse PINN that beats the classical estimator 6× on parameter bias with scatter at the Cramér–Rao bound on synthetic data. Part 6 is the transfer test; §3 is the road to a claim about real hardware.

**What the project can defensibly say today**, and it is worth writing this down precisely because the temptation is to overstate it:

- On **real** Li-ion data: heat generation as a function of depth of discharge, recovered from surface temperature alone, matching the textbook LCO profile and corroborated to 14 % by an independent instrument.
- On **synthetic** data: internal field reconstruction with the core-surface bow recovered to 0.3–3.7 %, and parameter bias reduced 6× against a classical rival.
- **Not yet**: internal field reconstruction validated on a real cell. That gap is closed by §3.2, not by more computation.